In [0]:
from pyspark.sql import functions as F
import re

LANDING = "/Volumes/workspace/easymoney_bronze/landing"
BRONZE = "workspace.easymoney_bronze"

TABLES = [
    "customer_commercial_activity",
    "customer_products",
    "customer_sociodemographics",
    "sales",
    "product_description",
]

def clean_col(name: str) -> str:
    """Normaliza nombres de columna: minúsculas, sin espacios ni caracteres raros."""
    name = re.sub(r"[^0-9a-z_]", "_", name.strip().lower())
    return name or "_unnamed"

def ingest_to_bronze(table: str) -> int:
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)  # en bronze todo entra como string
        .csv(f"{LANDING}/{table}.csv")
        .withColumn("_source_file", F.col("_metadata.file_path"))
    )
    df = df.toDF(*[clean_col(c) for c in df.columns])
    df = df.withColumn("_ingested_at", F.current_timestamp())

    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", True)
       .saveAsTable(f"{BRONZE}.{table}"))

    return spark.table(f"{BRONZE}.{table}").count()

In [0]:
for t in TABLES:
    n = ingest_to_bronze(t)
    print(f"{t:<32} {n:>12,} filas")

In [0]:
%sql
SHOW TABLES IN workspace.easymoney_bronze;

In [0]:
   %sql
   SELECT * FROM workspace.easymoney_bronze.customer_products LIMIT 5;

In [0]:
   %sql
   SELECT * FROM workspace.easymoney_bronze.customer_sociodemographics LIMIT 5;